# Logistic Regression

Classificador linear probabilistico.

- **Binario**: $P(y=1|x) = \sigma(w^T x + b)$
- **Multiclasse**: softmax sobre $K$ scores lineares.

Treinado minimizando cross-entropy via gradient descent, com L2 opcional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report,
)

In [ ]:
class LogisticRegression:
    """
    Regressao logistica via gradient descent.

    - Binaria: sigmoide + binary cross-entropy.
    - Multiclasse: softmax + cross-entropy (one-vs-rest implicito no softmax).

    Suporta L2 (ridge) e padronizacao opcional das features.
    """

    def __init__(self, lr=0.1, n_iters=1000, l2=0.0, scale=True, tol=1e-6):
        if lr <= 0:
            raise ValueError("lr must be positive.")
        if n_iters <= 0:
            raise ValueError("n_iters must be positive.")
        if l2 < 0:
            raise ValueError("l2 must be non-negative.")

        self.lr = lr
        self.n_iters = n_iters
        self.l2 = l2
        self.scale = scale
        self.tol = tol

        self.W = None
        self.b = None
        self.classes_ = None
        self.mean_ = None
        self.std_ = None
        self.loss_history_ = []

    @staticmethod
    def _sigmoid(z):
        # clipping evita overflow em exp
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    @staticmethod
    def _softmax(Z):
        Z = Z - Z.max(axis=1, keepdims=True)
        expZ = np.exp(Z)
        return expZ / expZ.sum(axis=1, keepdims=True)

    def _validate_X_y(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).ravel()
        if X.ndim != 2:
            raise ValueError("X must be 2D.")
        if len(X) != len(y):
            raise ValueError("X and y must have the same length.")
        if np.isnan(X).any():
            raise ValueError("X contains NaN values.")
        return X, y

    def _standardize_fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        return (X - self.mean_) / self.std_

    def _standardize(self, X):
        return (X - self.mean_) / self.std_

    def fit(self, X, y):
        X, y = self._validate_X_y(X, y)
        if self.scale:
            X = self._standardize_fit(X)

        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape
        n_classes = len(self.classes_)

        # indices one-hot para multiclass
        if n_classes > 2:
            Y = np.zeros((n_samples, n_classes))
            for idx, cls in enumerate(self.classes_):
                Y[y == cls, idx] = 1.0
            self.W = np.zeros((n_features, n_classes))
            self.b = np.zeros(n_classes)
        else:
            # binaria: mapeia classes para {0, 1}
            Y = (y == self.classes_[1]).astype(float)
            self.W = np.zeros(n_features)
            self.b = 0.0

        self.loss_history_ = []
        prev_loss = None

        for _ in range(self.n_iters):
            if n_classes > 2:
                logits = X @ self.W + self.b
                probs = self._softmax(logits)
                error = probs - Y

                grad_W = (X.T @ error) / n_samples + self.l2 * self.W
                grad_b = error.mean(axis=0)

                # cross-entropy
                loss = -np.mean(np.sum(Y * np.log(probs + 1e-12), axis=1))
            else:
                logits = X @ self.W + self.b
                probs = self._sigmoid(logits)
                error = probs - Y

                grad_W = (X.T @ error) / n_samples + self.l2 * self.W
                grad_b = error.mean()

                loss = -np.mean(
                    Y * np.log(probs + 1e-12) + (1 - Y) * np.log(1 - probs + 1e-12)
                )

            self.W -= self.lr * grad_W
            self.b -= self.lr * grad_b
            self.loss_history_.append(float(loss))

            if prev_loss is not None and abs(prev_loss - loss) < self.tol:
                break
            prev_loss = loss

        return self

    def predict_proba(self, X):
        if self.W is None:
            raise ValueError("Call fit() before predict_proba().")
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        if self.scale:
            X = self._standardize(X)

        if len(self.classes_) > 2:
            return self._softmax(X @ self.W + self.b)
        p1 = self._sigmoid(X @ self.W + self.b)
        return np.column_stack([1 - p1, p1])

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [ ]:
# Caso binario: breast cancer
data = datasets.load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(lr=0.1, n_iters=3000, l2=0.01, scale=True)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot()
plt.title("Logistic Regression - Breast Cancer")
plt.show()

plt.figure()
plt.plot(clf.loss_history_)
plt.title("Training loss")
plt.xlabel("iteration")
plt.ylabel("cross-entropy")
plt.show()

In [ ]:
# Caso multiclasse: iris
iris = datasets.load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
multi = LogisticRegression(lr=0.1, n_iters=3000, l2=0.01, scale=True)
multi.fit(X_tr, y_tr)
print("Multiclass accuracy:", multi.score(X_te, y_te))